# PRACTICE 1: CLASSIFYING SPAM EMAILS
## PHẦN 2: TIỀN XỬ LÝ DỮ LIỆU & KỸ THUẬT ĐẶC TRƯNG (DATA PREPROCESSING & FEATURE ENGINEERING)

> **Quy trình chuẩn hóa 5 bước khép kín:**
> 1. **Tiền xử lý & Khử trùng lặp:** Đọc 39.154 email thô CEAS_08, điền khuyết an toàn, làm sạch văn bản NLP (`clean_text`) và khử trùng lặp nội dung sạch (triệt tiêu rò rỉ Zero Duplicate Leakage 100%).
> 2. **Trích xuất 7 đặc trưng miền:** Tạo các thuộc tính hành vi email (`caps_ratio`, `exclamation_count`, `dollar_count`, `has_re_prefix`, `subject_exclamation`, `text_len`, `log_text_len`).
> 3. **Phân chia Train/Test 80/20 phân tầng:** Phân tách dữ liệu bảo toàn tỷ lệ nhãn (`stratify=y`, `seed=42`), kiểm chứng số văn bản trùng ranh giới = 0.
> 4. **Thiết lập ColumnTransformer đa tầng:** Kết hợp song song nhánh Text TF-IDF + Chi2 (2.000 chiều) và nhánh Số MinMaxScaler (7 chiều) thành không gian 2.007 chiều.
> 5. **Lưu trữ hiện vật:** Xuất `clean_emails.csv`, `train.csv`, `test.csv`, `train_test_data.joblib`, `preprocessor.joblib` bàn giao cho Chặng Huấn Luyện (Notebook 3).

---
## Khai báo Thư viện & Thiết lập Môi trường

* **Mục tiêu:** Nạp đầy đủ các thư viện khoa học dữ liệu và module tái sử dụng từ `data.scripts` và `scripts`.

In [2]:
# Khai báo thư viện & Thiết lập môi trường
import os
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    pass

# Đảm bảo import từ thư mục gốc dự án
ROOT_DIR = os.path.abspath(os.path.join(os.getcwd(), '..'))
if ROOT_DIR not in sys.path:
    sys.path.insert(0, ROOT_DIR)

from data.scripts.eda import load_raw_data
from data.scripts.data_preprocessing import handle_missing_values, clean_text
from scripts.feature_engineering import (
    clean_raw_email_data,
    extract_domain_features,
    split_stratified_train_test,
    build_column_transformer,
    export_train_test_csv,
    save_pipeline_and_splits
)

RANDOM_SEED = 42
print("Khởi tạo môi trường Tiền xử lý & Kỹ thuật đặc trưng thành công.")

Khởi tạo môi trường Tiền xử lý & Kỹ thuật đặc trưng thành công.


---
## Mục 1: Tiền Xử Lý Dữ Liệu Thô & Khử Trùng Lặp (Zero Duplicate Leakage)

> Khâu tiền xử lý thực hiện hoàn toàn độc lập từng dòng (row-level), không học tham số thống kê tổng thể, đảm bảo tính khách quan trước khi chia tập dữ liệu.

### 1.1 Nạp Dữ Liệu Thô CEAS_08 & Xử Lý Giá Trị Khuyết Thiếu
* **Mục tiêu:** Nạp 39.154 email thô, điền chuỗi rỗng cho `subject` và `body`, điền `'unknown'` cho `receiver` bảo đảm an toàn cho các phép tính toán chuỗi tiếp theo.

In [3]:
# 1. Nạp dữ liệu thô CEAS_08.csv và điền khuyết thiếu an toàn
raw_csv_path = os.path.join('..', 'data', 'raw', 'CEAS_08.csv')
df_raw = load_raw_data(data_path=raw_csv_path)
print(f"Quy mô dữ liệu thô ban đầu: {len(df_raw):,} dòng x {df_raw.shape[1]} cột")

df = handle_missing_values(df_raw)
print("\nĐối soát giá trị khuyết thiếu sau xử lý:")
print(f"- subject còn khuyết:  {df['subject'].isnull().sum()} dòng")
print(f"- body còn khuyết:     {df['body'].isnull().sum()} dòng")
print(f"- receiver còn khuyết: {df['receiver'].isnull().sum()} dòng")

Quy mô dữ liệu thô ban đầu: 39,154 dòng x 7 cột

Đối soát giá trị khuyết thiếu sau xử lý:
- subject còn khuyết:  0 dòng
- body còn khuyết:     0 dòng
- receiver còn khuyết: 0 dòng


### 1.2 Chuẩn Hóa Văn Bản NLP (`clean_text`) & Khử Trùng Lặp Nội Dung Sạch (Zero Duplicate Leakage)

* **Mục tiêu:** Lột bỏ hoàn toàn các lớp ngụy trang HTML, URL và từ dừng Stopwords để lộ nội dung cốt lõi của email.
* **Nguyên tắc Zero Duplicate Leakage:** Loại bỏ triệt để các mẫu có `text_clean` trùng nhau trước khi chia Train/Test, bảo đảm cùng một nội dung thư không xuất hiện ở cả 2 tập.

In [4]:
# 2. Tiền xử lý văn bản NLP bằng clean_text()
df['text_clean'] = (df['subject'] + ' ' + df['body']).apply(clean_text)

# Khử trùng lặp nội dung văn bản sạch (Nguyên tắc Zero Duplicate Leakage)
n_before = len(df)
df = df.drop_duplicates(subset=['text_clean']).reset_index(drop=True)
n_removed = n_before - len(df)

print(f"Số lượng email trước khử trùng: {n_before:,}")
print(f"Số lượng email trùng bị loại:   {n_removed:,}")
print(f"Số lượng email sạch thu được:   {len(df):,}")

label_dist = df['label'].value_counts()
print(f"\nPhân bố nhãn tập dữ liệu sạch: Ham={label_dist.get(0, 0):,} ({label_dist.get(0, 0)/len(df)*100:.2f}%), Spam={label_dist.get(1, 0):,} ({label_dist.get(1, 0)/len(df)*100:.2f}%)")

Số lượng email trước khử trùng: 39,154
Số lượng email trùng bị loại:   6,494
Số lượng email sạch thu được:   32,660

Phân bố nhãn tập dữ liệu sạch: Ham=17,094 (52.34%), Spam=15,566 (47.66%)


### 1.3 Lưu Trữ Dữ Liệu Sạch Chuẩn Hóa (`clean_emails.csv`)

* **Mục tiêu:** Xuất 32.660 email sạch sang tệp `clean_emails.csv` phục vụ kiểm tra và đối soát trực quan.

In [5]:
# 3. Xuất tệp dữ liệu sạch đã chuẩn hóa clean_emails.csv
clean_path = os.path.join('..', 'data', 'processed', 'clean_emails.csv')
os.makedirs(os.path.dirname(clean_path), exist_ok=True)
cols = ['sender', 'receiver', 'date', 'subject', 'body', 'text_clean', 'urls', 'label']
cols_to_save = [c for c in cols if c in df.columns]
df[cols_to_save].to_csv(clean_path, index=False, encoding='utf-8')
print(f"✔ Đã lưu thành công {len(df):,} dòng sạch tại: {clean_path}")

✔ Đã lưu thành công 32,660 dòng sạch tại: ..\data\processed\clean_emails.csv


---
## Mục 2: Kỹ Thuật Trích Xuất Đặc Trưng Miền & Biến Đổi (Domain Feature Engineering)

### 2.1 Trích Xuất 7 Đặc Trưng Miền Nghiệp Vụ (Domain Feature Extraction)
* **Mục tiêu:** Dựa trên phát hiện từ EDA (Notebook 1), trích xuất 7 đặc trưng số bổ trợ cho bộ từ vựng TF-IDF:
  1. `text_len`: Độ dài ký tự của chuỗi văn bản sạch.
  2. `log_text_len`: $\ln(1 + \text{text\_len})$ nén phân phối lệch phải cực đoan.
  3. `caps_ratio`: Tỷ lệ ký tự viết hoa trong văn bản gốc.
  4. `exclamation_count`: Số dấu chấm than (!).
  5. `dollar_count`: Số ký hiệu tiền tệ ($).
  6. `has_re_prefix`: Cờ có tiền tố đối thoại 'Re:' ở tiêu đề hay không.
  7. `subject_exclamation`: Cờ tiêu đề có chứa dấu (!) hay không.

In [6]:
# 1. Trích xuất 7 đặc trưng miền qua hàm chuẩn hóa từ scripts.feature_engineering
df = extract_domain_features(df)

# Hiển thị mẫu dữ liệu 7 đặc trưng miền trích xuất
feature_preview = ['log_text_len', 'caps_ratio', 'exclamation_count', 'dollar_count', 'has_re_prefix', 'subject_exclamation', 'label']
print("Mẫu 5 dòng đặc trưng miền trích xuất:")
display(df[feature_preview].head())


Mẫu 5 dòng đặc trưng miền trích xuất:


,log_text_len,caps_ratio,exclamation_count,dollar_count,has_re_prefix,subject_exclamation,label
0,4.983607,0.023411,2,0,0,0,1
1,4.025352,0.038095,0,0,0,0,1
2,6.934397,0.165017,0,0,0,0,1
3,8.852236,0.008018,4,461,1,0,0
4,4.127134,0.198020,0,0,0,0,1


---
### 2.2 Biến Đổi Nén Phân Phối Lệch & Phân Nhóm Tứ Phân Vị (Log Transform & Quantile Binning)

* **Mục tiêu:** Tạo cột độ dài tiêu đề `subject_len` và áp dụng $\ln(1 + \text{text\_len})$ nén phân phối đuôi dài, triệt tiêu ảnh hưởng của kỹ thuật Bayesian Poisoning.

In [7]:
# 2. Tạo cột độ dài tiêu đề và chia 4 khoảng tứ phân vị cho độ dài văn bản
df['subject_len'] = df['subject'].fillna('').astype(str).str.len()
df['length_bin'] = pd.qcut(df['text_len'], q=4, labels=['Ngắn', 'Trung bình', 'Dài', 'Rất dài'])

print("Phân bố số lượng email theo 4 nhóm độ dài văn bản:")
display(df['length_bin'].value_counts().rename("Số lượng email"))


Phân bố số lượng email theo 4 nhóm độ dài văn bản:


length_bin
Ngắn          8221
Dài           8166
Rất dài       8163
Trung bình    8110
Name: Số lượng email, dtype: int64

---
### 2.3 Kiểm Chứng Ý Nghĩa Phân Phối Thống Kê Giữa Thư Thường (Ham) & Thư Rác (Spam)

* **Mục tiêu:** Tính toán Mean, Median, Std của 7 đặc trưng miền phân tách theo nhãn `label` (0: Ham, 1: Spam) để xác nhận tính phân biệt thống kê.

In [8]:
# 3. Bảng thống kê phân phối (mean, median, std) theo nhãn Ham (0) vs Spam (1)
domain_stats_cols = ['caps_ratio', 'exclamation_count', 'dollar_count', 'text_len', 'log_text_len', 'has_re_prefix', 'subject_exclamation']
domain_distribution = df.groupby('label')[domain_stats_cols].agg(['mean', 'median', 'std']).round(4).T

print("Bảng thống kê phân phối đặc trưng miền theo nhãn (Ham vs Spam):")
display(domain_distribution)


Bảng thống kê phân phối đặc trưng miền theo nhãn (Ham vs Spam):


label                               0         1
caps_ratio          mean       0.0519    0.0535
                    median     0.0431    0.0323
                    std        0.0344    0.0566
exclamation_count   mean       1.0676    1.1329
                    median     0.0000    0.0000
                    std       10.3735    1.6503
dollar_count        mean       1.2244    0.1296
                    median     0.0000    0.0000
                    std        8.5561    0.6623
text_len            mean    1434.1327  212.0428
                    median   683.0000  163.0000
                    std     2896.0223  277.3929
log_text_len        mean       6.6430    5.0520
                    median     6.5280    5.0999
                    std        1.0119    0.7195
has_re_prefix       mean       0.5526    0.0075
                    median     1.0000    0.0000
                    std        0.4972    0.0860
subject_exclamation mean       0.0217    0.1106
                    median     0.0000    0.0000
                    std        0.1457    0.3136

---
## Mục 3: Phân Chia Train/Test 80/20 Phân Tầng Chống Rò Rỉ Dữ Liệu

> **Nguyên tắc Zero Data Leakage:** Mọi phép tính toán học (fit TF-IDF, Chi2, MinMaxScaler) chỉ được phép học trên tập Train. Tuyệt đối không fit trên toàn bộ dữ liệu trước khi chia tách.

### 3.1 Phân Chia Tập Train/Test 80/20 Phân Tầng (`stratify=y`)

* **Mục tiêu:** Chia tách 80% Train ({train_size:,} mẫu) và 20% Test ({test_size:,} mẫu) bảo toàn tỷ lệ nhãn tự nhiên 47.7% Spam, với `random_state = 42`.

In [9]:
# Phân chia Train/Test 80/20 phân tầng chống rò rỉ dữ liệu qua hàm chuẩn hóa
X_train, X_test, y_train, y_test = split_stratified_train_test(df, test_size=0.20, random_state=RANDOM_SEED)
print(f"Train: {len(X_train):,} mẫu (Tỷ lệ Spam: {y_train.mean()*100:.2f}%)")
print(f"Test:  {len(X_test):,} mẫu (Tỷ lệ Spam: {y_test.mean()*100:.2f}%)")

# Kiểm tra trùng lặp ranh giới (Zero Data Leakage Check)
train_texts = set(X_train['text_clean'].dropna().astype(str))
test_texts = set(X_test['text_clean'].dropna().astype(str))
overlap = len(train_texts.intersection(test_texts))
print(f"\nKiểm tra trùng lặp văn bản Train/Test: {overlap} mẫu trùng (Zero Data Leakage 100%)")

Train: 26,128 mẫu (Tỷ lệ Spam: 47.66%)
Test:  6,532 mẫu (Tỷ lệ Spam: 47.66%)

Kiểm tra trùng lặp văn bản Train/Test: 0 mẫu trùng (Zero Data Leakage 100%)


---
## Mục 4: Thiết Lập Dây Chuyền Tiền Xử Lý Đa Tầng (ColumnTransformer)

> Dây chuyền gồm 2 nhánh xử lý song song:
> 1. **Nhánh văn bản (`text_clean`):** `TfidfVectorizer(max_features=5000, ngram=(1,2))` + `SelectKBest(chi2, k=2000)` chọn lọc đúng 2.000 từ khóa tương quan cao nhất.
> 2. **Nhánh số học (7 đặc trưng miền):** `MinMaxScaler()` co giãn về miền $[0, 1]$, triệt tiêu số âm bảo vệ thuật toán Naive Bayes.

In [10]:
# Xây dựng Pipeline Tiền xử lý Đa tầng qua hàm chuẩn hóa build_column_transformer()
preprocessor = build_column_transformer(max_features=5000, k_best=2000, ngram_range=(1, 2))


---
## Mục 5: Đóng Gói Hiện Vật & Bàn Giao Dây Chuyền Tiền Xử Lý

* **Mục tiêu:** Xuất độc lập 2 tệp CSV (`train.csv`, `test.csv`) có thể click mở xem bằng Excel/Notepad và lưu nhị phân `train_test_data.joblib`, `preprocessor.joblib` phục vụ Chặng Huấn Luyện (Notebook 3).

In [11]:
# Đóng gói preprocessor.joblib, train_test_data.joblib và xuất train.csv, test.csv
prep_path, data_path = save_pipeline_and_splits(
    preprocessor,
    X_train, X_test, y_train, y_test,
    export_csv=True
)
print(f"✔ Đã lưu Pipeline tiền xử lý: {prep_path}")
print(f"✔ Đã lưu Dữ liệu phân chia:     {data_path}")
print("✔ Đã xuất độc lập: data/processed/train.csv & test.csv")

✔ Đã lưu Pipeline tiền xử lý: D:\DH_GTVT\NĂM 3\ML\baitap_code_Nhom_3\PRACTICE 1_CLASSIFYING_SPAM_EMAILS\models\saved_models\preprocessor.joblib
✔ Đã lưu Dữ liệu phân chia:     D:\DH_GTVT\NĂM 3\ML\baitap_code_Nhom_3\PRACTICE 1_CLASSIFYING_SPAM_EMAILS\data\processed\train_test_data.joblib
✔ Đã xuất độc lập: data/processed/train.csv & test.csv
